# ANALYSE-06 : Sensibilité 2026 — Huang, le décideur qui réfute, et ce que le dépôt openai/math a bougé

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-05b — KLS constant O(1)](ANALYSE-05b-KLS-Song-Zhang-O1.ipynb) | [ANALYSE-08 — Ramsey / van der Waerden](ANALYSE-08-Ramsey-VdW.ipynb) |

**Version anglaise** : [ANALYSE-06 (en)](ANALYSE-06-Sensibilite-2026-Lean-Python_en.ipynb)

**Série** : [README du dossier ANALYSE](README.md) · Lac : [`sensitivity_lean`](../sensitivity_lean/) · Carnets frères : [Lean-12](../Lean-12-Sensitivity-Theorem.ipynb), [Lean-12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb), [Lean-12c](../Lean-12c-Tensor-Product-Representations-Lean.ipynb)

Ce carnet digère le volet « sensibilité » de la publication [openai/math](https://github.com/openai/math) (2026) du point de vue de **ce que nous savons vérifier nous-mêmes**. Il part du théorème de Huang — **nôtre**, porté par le lac `sensitivity_lean` et vérifié ici par `#check` et `#print axioms` — puis cite les familles **127, 132, 186, 192** du dépôt **avec leur statut épistémique**, puis prouve et mesure **nous-mêmes** les petites séparations : définitions `sensitivity`/`blockSensitivity`, témoin `tribes32` (ajout additif, fichier neuf), expériences Python exactes.

Le fil directeur est la méthode du dépôt lui-même, retournée vers nous : **mesuré, pas supposé**. Deux fois dans ce carnet, une intuition raisonnable d'auteur est réfutée par un instrument — une fois par le noyau Lean (`decide`), une fois par la mesure Python exhaustive. Les deux récits sont consignés dans la friction (§7).

## 1. Énoncé — la conjecture de sensibilité (1992–2019)

Soit $f : \{0,1\}^n \to \{0,1\}$ une fonction booléenne **totale**. Deux quantités mesurent sa « rugosité » locale sur le cube :

- la **sensibilité** $s(f, x)$ en un sommet $x$ : le nombre de coordonnées dont le retournement (une à une) change $f(x)$ ; $s(f) = \max_x s(f,x)$ ;
- la **sensibilité de bloc** $bs(f, x)$ en $x$ : le nombre **maximal de blocs disjoints** dont le retournement simultané change chacun $f(x)$ ; $bs(f) = \max_x bs(f,x)$.

Un singleton est un bloc : $s(f) \le bs(f)$ toujours. La conjecture de sensibilité (Nisan 1992 ; Nisan–Szegedy 1994) demandait si la réciproque tient à un polynôme près :

> Existe-t-il un polynôme $P$ tel que $bs(f) \le P(s(f))$ pour toute $f$ totale ?

La réponse est **oui** — c'est le théorème de Huang (2019), qui donne la borne explicite :

$$bs(f) \;\le\; 2 \cdot s(f)^4.$$

Ce n'est pas une borne serrée : l'exposant exact est ouvert, coincé entre 2 (construction de Rubinstein 1995, raffinée par la famille 132 du dépôt openai/math) et 4 (Huang). C'est précisément cette bande $[2, 4]$ que le carnet parcourt, en citant chaque étage avec son statut.

## 2. La chaîne de la preuve — trois maillons, un seul à nous

La borne $bs \le 2s^4$ n'est pas prouvée d'un bloc : c'est une chaîne de trois maillons, chacun avec sa propre histoire — et un seul est dans notre lac.

| # | Maillon | Énoncé | Statut |
|---|---|---|---|
| 1 | **Théorème de degré (Huang 2019)** | tout sous-graphe induit $H$ de $Q_n$ avec $\vert H \vert > 2^{n-1}$ possède un sommet de degré intérieur $\ge \sqrt{n}$ | **nôtre** — `huang_degree_theorem`, lac `sensitivity_lean` ([MainTheorem.lean](../sensitivity_lean/Sensitivity/MainTheorem.lean)), vérifié au §3 |
| 2 | **Équivalence de Gotsman–Linial (1991)** | le maillon 1 équivaut à : $\deg(f) \le s(f)^2$ pour toute $f$ totale | **cité, non prouvé** — pont combinatoire externe |
| 3 | **Nisan–Szegedy (1994)** | $bs(f) \le 2 \cdot \deg(f)^2$ pour toute $f$ totale | **cité, non prouvé** |

Maillons 2 et 3 enchaînés : $bs(f) \le 2\deg(f)^2 \le 2(s(f)^2)^2 = 2s(f)^4$.

Pourquoi le lac ne contient que le maillon 1 ? Parce que c'est lui le théorème **difficile** — l'argument spectral ($\sqrt{n}$ comme valeur propre max d'une matrice signée de Cauchy) — et parce que les maillons 2 et 3 sont des réductions combinatoires dont la formalisation complète est hors d'échelle du lac. Le lac porte donc : le maillon 1 complet, **plus** (ajout 2026) les définitions et les séparations locales calculables — l'objet exact que les familles 127/132/192 du dépôt manipulent.

## 3. Le socle Lean — d'abord, un audit du pin

Avant de lire une ligne du lac : **qu'existe-t-il déjà dans Mathlib au pin de ce lac ?** Le lac `sensitivity_lean` tourne sous le toolchain `v4.33.0`, Mathlib épinglé `db584cd6d46c` — le rev **majoritaire** du dépôt (contrairement au lac du carnet 05-KLS, qui introduisait un rev minoritaire). L'audit sonde Mathlib propre et l'Archive du paquet.

In [1]:
# Audit Mathlib au pin v4.33.0 (db584cd6d46c) : les notions du chainon existent-elles ?
import subprocess, os
# La distro WSL par defaut peut etre docker-desktop (sans bash) : cible explicite,
# surchargeable par WSL_DISTRO (convention scripts/lean/setup_lean4_all.py).
WSL = ["wsl", "-d", os.environ.get("WSL_DISTRO", "Ubuntu")]
LAKE_DIR = subprocess.run(WSL + ["wslpath", "-a", os.getcwd().replace("\\", "/")],
                          capture_output=True, text=True,
                          encoding="utf-8", errors="replace").stdout.strip() + "/../sensitivity_lean"
MATHLIB = LAKE_DIR + "/.lake/packages/mathlib"

def grep_count(pattern, where):
    cmd = "grep -rli --include='*.lean' -m1 -E '" + pattern + "' " + where + " 2>/dev/null | wc -l"
    r = subprocess.run(WSL + ["-e", "bash", "-lc", cmd], capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    return int(r.stdout.strip() or 0)

def count_lean_files(where):
    cmd = "find " + where + " -name '*.lean' 2>/dev/null | wc -l"
    r = subprocess.run(WSL + ["-e", "bash", "-lc", cmd], capture_output=True,
                       text=True, encoding="utf-8", errors="replace")
    return int(r.stdout.strip() or 0)

N_FILES = count_lean_files(MATHLIB + '/Mathlib')
notions = [
    ("sensitivity / blockSensitivity (fonction booleenne)", "sensitivity|blockSensitivity"),
    ("influence d'une variable (cube booleen)",              "influence.*[Bb]ool"),
    ("degre d'une fonction booleenne",                       "booleanDegree|BooleanDegree"),
    ("coefficients de Fourier du cube booleen",              "fourierCoefficient|Fourier.*[Cc]ube"),
]
print(f"Fichiers .lean inspectes dans Mathlib/ : {N_FILES}")
print(f"{'Notion (chainon 1-3)':<52s} {'fichiers dans Mathlib/':>22s}")
print("-" * 76)
for label, pat in notions:
    print(f"{label:<52s} {grep_count(pat, MATHLIB + '/Mathlib'):>22d}")
# et l'Archive du paquet ?
r = subprocess.run(WSL + ["-e", "bash", "-lc", "ls " + MATHLIB + "/Archive/Sensitivity.lean 2>&1"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
# La sortie de ls est un chemin absolu dependant du poste (classe machine-path,
# EPIC #11044) : seul le suffixe relatif a la racine du paquet mathlib est imprime.
archive = r.stdout.strip()
print()
print("Archive/Sensitivity.lean (racine du paquet) :",
      archive[len(MATHLIB):] if archive.startswith(MATHLIB) else archive)

Fichiers .lean inspectes dans Mathlib/ : 8311
Notion (chainon 1-3)                                 fichiers dans Mathlib/
----------------------------------------------------------------------------


sensitivity / blockSensitivity (fonction booleenne)                       0


influence d'une variable (cube booleen)                                   0


degre d'une fonction booleenne                                            0


coefficients de Fourier du cube booleen                                   0

Archive/Sensitivity.lean (racine du paquet) : /Archive/Sensitivity.lean


### Lecture de l'audit

**Zéro dans Mathlib propre, un fichier dans l'Archive.** Au pin `v4.33.0`, Mathlib n'a ni sensibilité, ni sensibilité de bloc, ni influence, ni degré booléen : ces objets vivent dans la recherche, pas dans la librairie. L'exception est `Archive/Sensitivity.lean` — **à la racine du paquet**, hors de l'espace de noms `Mathlib.*` — la formalisation du théorème de Huang par Reid Barton, Johan Commelin, Jesse Michael Han, Chris Hughes, Robert Y. Lewis et Patrick Massot.

Le lac `sensitivity_lean` est précisément le **dust-off autonome** de cette Archive : le théorème de Huang extrait dans son propre workspace Lake, licence Apache 2.0 conservée (voir [`NOTICE.md`](../sensitivity_lean/NOTICE.md)). Ce que l'Archive ne porte pas — et que le lac ajoute — : les définitions `sensitivity`/`blockSensitivity` au sens de Nisan–Szegedy et leurs séparations calculables (`Separations.lean`, 2026), auxquelles ce carnet ajoute `Sens2026.lean`. L'audit se conclut donc à l'inverse du carnet 05 : le maillon difficile **existait** en amont, notre travail est l'extraction, l'extension additive et la vérification.

In [2]:
# Compilation reelle du lac sensitivity_lean (WSL + lake, toolchain v4.33.0)
# + compteur canonique de sorry du depot (distinct_code_sorry, pas grep -c).
import subprocess, sys
r = subprocess.run(WSL + ["-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && lake build 2>&1 | tail -3"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace",
                   timeout=600)
print(r.stdout)
print("stderr:", r.stderr.strip() or "(vide)")
print("rc =", r.returncode)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
LAKE_WIN = os.path.abspath(os.path.join(os.getcwd(), "..", "sensitivity_lean"))
r2 = subprocess.run([sys.executable, os.path.join(ROOT, "scripts", "lean", "count_code_sorry.py"),
                     "--json", "--lake", LAKE_WIN],
                    capture_output=True, text=True, encoding="utf-8", errors="replace",
                    timeout=120)
print("compteur canonique --lake sensitivity_lean :")
print(r2.stdout.strip()[-700:] or r2.stderr.strip()[-700:])

Build completed successfully (3025 jobs).

stderr: (vide)
rc = 0


compteur canonique --lake sensitivity_lean :
{
  "lakes": [
    {
      "lake": "MyIA.AI.Notebooks/SymbolicAI/Lean/sensitivity_lean",
      "files": 22,
      "naive_sorry": 1,
      "code_sorry": 0,
      "distinct_code_sorry": 0,
      "vacuous": []
    }
  ]
}


### Lecture du résultat

`Build completed successfully` : le lac compile en entier — racines agrégatrices `Sensitivity.lean` (FR) et `Sensitivity_en.lean` (convention i18n #4980), modules `Sensitivity/*` et leurs jumeaux `_en`, 0 erreur. Et **aucun `sorry` de code** : la mesure est celle du compteur canonique du dépôt (`count_code_sorry.py --lake`, champ `distinct_code_sorry`), exécuté ci-dessus — le comptage naïf `grep -c sorry` est proscrit pour cette mesure (il compte la prose des docstrings ; voir `.claude/rules/anti-regression.md`).

Outillage : toolchain `leanprover/lean4:v4.33.0`, Mathlib épinglé `db584cd6d46c` — le rev **partagé par la majorité des lacs du dépôt** : ce carnet n'introduit **aucun** rev Mathlib supplémentaire en CI, là où le lac KLS du carnet 05 en introduisait un.

In [3]:
# Lecture du socle : definitions de Separations.lean + ajout Sens2026.lean (extraits numerotes)
r = subprocess.run(WSL + ["-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && cat -n Sensitivity/Separations.lean | sed -n '33,57p'"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout)
print("=" * 70)
r = subprocess.run(WSL + ["-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && cat -n Sensitivity/Sens2026.lean | sed -n '44,68p'"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout)

    33	/-- Sensibilité locale de `F` en `x` : nombre de coordonnées dont le
    34	retournement (une à une) change la valeur de `F`. -/
    35	def sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ :=
    36	  (Finset.univ.filter fun i => F (flip x i) ≠ F x).card
    37	
    38	/-- Retournement simultané d'un **bloc** `S` de coordonnées de `x`. -/
    39	def flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n :=
    40	  fun j => if j ∈ S then !(x j) else x j
    41	
    42	/-- Sensibilité de bloc locale de `F` en `x` — **définition standard**
    43	(Nisan–Szegedy) : le **maximum du nombre de blocs sensibles deux à deux
    44	disjoints**, c'est-à-dire le cardinal maximal d'une famille de blocs **non
    45	vides** dont le retournement simultané change la valeur de `F`, et dont deux
    46	membres distincts sont toujours d'intersection vide.
    47	
    48	Ce n'est **pas** le compte de tous les blocs sensibles : au sommet nul de `or3`,
    49	les sept blocs non vides sont tous sens

    44	open Bool Finset Fintype
    45	
    46	/-- Témoignage « tribus » en dimension 6 : disjonction de trois conjonctions
    47	de taille deux (trois « équipes » de deux coordonnées) — les `tribes` de la
    48	littérature Nisan–Szegedy, à l'échelle calculable. -/
    49	def tribes32 (x : Q 6) : Bool :=
    50	  (x 0 && x 1) || (x 2 && x 3) || (x 4 && x 5)
    51	
    52	/-- Au sommet nul, aucune coordonnée seule n'allume `tribes32` : il faut
    53	retourner une tribu **entière**. La sensibilité locale y est nulle. -/
    54	theorem tribes32_sensitivity_zero :
    55	    sensitivity tribes32 (fun _ => false) = 0 := by
    56	  decide
    57	
    58	/-- **La réfutation de l'intuition, par énumération exhaustive.** Au sommet
    59	« une lampe allumée par tribu » (coordonnées impaires), chaque première
    60	coordonnée allume sa conjonction à elle seule : la sensibilité locale vaut
    61	**3**, pas 2. Le majorant naïf « au plus une taille de tribu » est faux —
    62	c'est le noyau

In [4]:
# Verification formelle depuis le lac : signatures + axiomes (lake env lean Check.lean)
# Delai interne calibre par mesure : 240 s expiraient sur lac froid (sortie vide,
# kill parti dans stderr jamais imprime) ; 600 s couvrent le typecheck a froid
# sur IO 9p (cf. Lean-15e : 298 s mesurees pour un seul jeu de #check).
r = subprocess.run(WSL + ["-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && timeout 600 lake env lean Check.lean 2>&1"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=620)
if not r.stdout.strip():
    print("ECHEC verification formelle : stdout vide -- stderr/rc ci-dessous, jamais silencieux")
    print("stderr:", r.stderr.strip() or "(vide)")
    print("rc =", r.returncode)
else:
    print(r.stdout)

@Sensitivity.sensitivity : {n : ℕ} → (Sensitivity.Q n → Bool) → Sensitivity.Q n → ℕ
@Sensitivity.blockSensitivity : {n : ℕ} → (Sensitivity.Q n → Bool) → Sensitivity.Q n → ℕ
Sensitivity.tribes32 : Sensitivity.Q 6 → Bool
Sensitivity.tribes32_sensitivity_zero : (Sensitivity.sensitivity Sensitivity.tribes32 fun x => false) = 0
Sensitivity.tribes32_sensitivity_eq_three : (Sensitivity.sensitivity Sensitivity.tribes32 fun i =>
    decide (↑i % 2 = 1)) =
  3
Sensitivity.tribes32_blockSensitivity_ge_three : 3 ≤ Sensitivity.blockSensitivity Sensitivity.tribes32 fun x => false
Sensitivity.tribes32_local_separation : (Sensitivity.sensitivity Sensitivity.tribes32 fun x => false) <
  Sensitivity.blockSensitivity Sensitivity.tribes32 fun x => false
Sensitivity.or3_blockSensitivity : (Sensitivity.blockSensitivity Sensitivity.or3 fun x => false) = 3
Sensitivity.orOfAnds_strict_separation : (Sensitivity.sensitivity Sensitivity.orOfAnds fun x => false) <
  Sensitivity.blockSensitivity Sensitivity.orOfAnd

### Lecture des `#check` et `#print axioms`

Onze déclarations rendent leurs signatures **réelles** — élaborées par le compilateur au pin, pas recopiées :

- `huang_degree_theorem` — le maillon 1 de la chaîne, **nôtre** (dust-off Archive) : dans tout $H \subseteq Q_{m+1}$ de cardinal $\ge 2^m + 1$, il existe $q \in H$ avec $\sqrt{m+1} \le |H \cap q.\text{adjacent}|$ ;
- `sensitivity`, `blockSensitivity` — les définitions au sens de Nisan–Szegedy (la sensibilité de bloc comme **maximum de familles de blocs disjoints**, pas le compte de tous les blocs sensibles) ;
- `tribes32` et ses quatre théorèmes — l'ajout 2026 de ce carnet (§5) ;
- `or3_blockSensitivity`, `orOfAnds_strict_separation` — les témoins de `Separations.lean`.

Les sept `#print axioms` rendent **exactement les trois axiomes standard** — `propext, Classical.choice, Quot.sound` — y compris pour les preuves `by decide` : aucune dépendance cachée, aucun `sorry` transitif (`sorryAx` serait listé), aucun `native_decide`. C'est la définition opérationnelle du statut **« nôtre »** dans la table du §8.

## 4. Ce que le lac ne contient pas — et pourquoi

Les **maillons 2 et 3** de la chaîne (§2) ne sont pas dans le lac :

- l'**équivalence de Gotsman–Linial** (maillon 1 $\Leftrightarrow \deg \le s^2$) est une réduction combinatoire entre coloriages de cube et polynômes représentants — sa formalisation exige la théorie des polynômes multivariés sur $\mathbb{F}_2$ **au-delà** de ce que le lac pose ;
- la borne **Nisan–Szegedy** $bs \le 2\deg^2$ est elle-même une preuve combinatoire non triviale.

Le corollaire final $bs(f) \le 2s(f)^4$ n'est donc **pas formellement déductible dans le lac** : le carnet le cite avec son statut — théorème publié (Huang 2019, corollaire standard), maillon terminal non formalisé ici. Ce découpage honnête est le même que celui du carnet 05 pour la chaîne BKL : le lac porte le socle, la chaîne est contée avec ses statuts.

En revanche, ce que le dépôt openai/math appelle « les mêmes objets » — sensibilité, influences, Fourier du cube — est **littéralement** le vocabulaire de notre lac : c'est la résonance mesurée qui a motivé ce carnet (Epic Origami #19898, Pli 2).

## 5. Expérience 1 — mesurer ce que le `decide` a tranché

L'ajout 2026 du lac, `Sens2026.lean`, définit le témoin **tribus à trois équipes** sur $Q_6$ :

$$\texttt{tribes32}(x) = (x_0 \wedge x_1) \vee (x_2 \wedge x_3) \vee (x_4 \wedge x_5)$$

et prouve quatre énoncés par énumération exhaustive du noyau (`decide`) : $s(\mathbf{0}) = 0$, $s(x^{\text{impair}}) = 3$, $bs(\mathbf{0}) \ge 3$, donc la séparation locale $0 < 3$ au sommet nul.

La mesure Python ci-dessous est un **instrument indépendant** du noyau Lean : elle calcule $s$ et $bs$ **exacts** par programmation dynamique sur sous-ensembles de coordonnées (le $bs$ exact d'un sommet est un set-packing ; la DP $dp[\text{masque}]$ = maximum de blocs disjoints couverts par le masque le résout en $O(2^n \cdot |\text{blocs}|)$, là où la récursion naïve explose dès $n = 6$ — friction §7). Mesurer le même objet par deux instruments indépendants qui rendent les mêmes chiffres, c'est la croisée minimale de vérification.

In [5]:
# Experience 1 : s et bs exacts (DP sous-ensembles) pour les temoins du lac, global et local.
from itertools import product

def flip(x, i):
    return tuple(1 - v if j == i else v for j, v in enumerate(x))

def s_at(f, n, x):
    return sum(1 for i in range(n) if f(flip(x, i)) != f(x))

def bs_at(f, n, x):
    # bs exact en x : DP sur sous-ensembles de coordonnees.
    # dp[mask] = max de blocs sensibles disjoints utilisables dans mask.
    fx = f(x)
    full = (1 << n) - 1
    sens = [B for B in range(1, full + 1)
            if f(tuple(1 - v if (B >> j) & 1 else v for j, v in enumerate(x))) != fx]
    by_low = {}
    for B in sens:
        by_low.setdefault(B & (-B), []).append(B)
    dp = [0] * (full + 1)
    for mask in range(1, full + 1):
        low = mask & (-mask)
        best = dp[mask ^ low]                       # coordonnee low laissee inutilisee
        for B in by_low.get(low, ()):               # bloc sensible couvrant low, inclus dans mask
            if B & mask == B:
                cand = 1 + dp[mask ^ B]
                if cand > best:
                    best = cand
        dp[mask] = best
    return dp[full]

def s(f, n):
    return max(s_at(f, n, x) for x in product((0, 1), repeat=n))

def bs(f, n):
    return max(bs_at(f, n, x) for x in product((0, 1), repeat=n))

def or3(x): return x[0] or x[1] or x[2]
def or_of_ands_4(x): return (x[0] and x[1]) or (x[2] and x[3])
def tribes32(x): return (x[0] and x[1]) or (x[2] and x[3]) or (x[4] and x[5])

for nom, f, n in [("or3 (n=3)", or3, 3), ("orOfAnds (n=4)", or_of_ands_4, 4),
                  ("tribes32 (n=6)", tribes32, 6)]:
    print(f"{nom:<18s} s global = {s(f, n)} | bs global = {bs(f, n)}")

z, impair = (0,) * 6, tuple(i % 2 for i in range(6))
print()
print("tribes32 local au sommet nul   : s =", s_at(tribes32, 6, z), "| bs =", bs_at(tribes32, 6, z))
print("tribes32 local au sommet impair: s =", s_at(tribes32, 6, impair))
print("or3 local au sommet nul        : bs =", bs_at(or3, 3, (0, 0, 0)),
      "(le lac prouve = 3 : or3_blockSensitivity)")
print("orOfAnds local au sommet nul   : bs =", bs_at(or_of_ands_4, 4, (0, 0, 0, 0)),
      "(le lac prouve seulement > 0 : orOfAnds_blockSensitivity_pos)")

or3 (n=3)          s global = 3 | bs global = 3
orOfAnds (n=4)     s global = 2 | bs global = 2
tribes32 (n=6)     s global = 3 | bs global = 3

tribes32 local au sommet nul   : s = 0 | bs = 3
tribes32 local au sommet impair: s = 3
or3 local au sommet nul        : bs = 3 (le lac prouve = 3 : or3_blockSensitivity)
orOfAnds local au sommet nul   : bs = 2 (le lac prouve seulement > 0 : orOfAnds_blockSensitivity_pos)


### Lecture du résultat

Chiffre par chiffre, la mesure Python **reproduit le verdict du noyau** — deux instruments indépendants, mêmes chiffres :

| Quantité | `decide` (lac) | DP Python |
|---|---|---|
| $s(\texttt{tribes32}, \mathbf{0})$ | $0$ | $0$ |
| $s(\texttt{tribes32}, x^{\text{impair}})$ | $3$ | $3$ |
| $bs(\texttt{tribes32}, \mathbf{0})$ | $\ge 3$ (prouvé) | $3$ (exact) |
| $bs(\texttt{or3}, \mathbf{0})$ | $= 3$ (prouvé) | $3$ |
| $bs(\texttt{orOfAnds}, \mathbf{0})$ | $> 0$ (prouvé) | $2$ (exact) |

Deux lectures fines. **Première** : les sensibilités *globales* valent toutes $s = bs$ pour ces témoins — les séparations prouvées au lac sont **locales** (en un sommet), pas globales ; c'est le sujet du §6. **Seconde** : pour `orOfAnds`, la mesure exacte ($bs(\mathbf{0}) = 2$) **dépasse** ce que le lac prouve ($> 0$) — normal et sain : le théorème formel le moins cher n'est pas l'énoncé exact ; la mesure comble l'écart à coût nul, et la preuve formelle certifie indépendamment ce qu'elle énonce. (Les blocs $\{0,1\}$ et $\{2,3\}$ sont tous deux sensibles au sommet nul et disjoints : $bs = 2$.)

In [6]:
# Arc tribes t=2, k croissant : bs exact par DP pour k <= 4, s seul au-dela (cout 2^n x 3^n).
def tribes_kt(k, t):
    n = k * t
    def f(x):
        return any(all(x[i * t + j] for j in range(t)) for i in range(k))
    return f, n

print(f"{'famille':<22s} {'s global':>9s} {'bs global':>10s} {'ratio':>6s}")
print("-" * 50)
for k in (2, 3, 4):
    fk, nk = tribes_kt(k, 2)
    sk, bsk = s(fk, nk), bs(fk, nk)
    print(f"tribes k={k} t=2 (n={nk:<2d})  {sk:>9d} {bsk:>10d} {bsk/sk:>6.2f}")
for k in (5, 6):
    fk, nk = tribes_kt(k, 2)
    print(f"tribes k={k} t=2 (n={nk:<2d})  {s(fk, nk):>9d} {'--':>10s} {'--':>6s}   (bs non mesure)")

famille                 s global  bs global  ratio
--------------------------------------------------
tribes k=2 t=2 (n=4 )          2          2   1.00
tribes k=3 t=2 (n=6 )          3          3   1.00


tribes k=4 t=2 (n=8 )          4          4   1.00
tribes k=5 t=2 (n=10)          5         --     --   (bs non mesure)


tribes k=6 t=2 (n=12)          6         --     --   (bs non mesure)


### Lecture de l'arc

**Ratio 1,00 partout.** Pour les tribus à $t = 2$, la sensibilité globale rattrape toujours la sensibilité de bloc : $s = bs = k$ (au sommet « une lampe allumée par tribu », chaque première coordonnée porte à elle seule une conjonction — le sommet $x^{\text{impair}}$ du `decide` — et au sommet nul les $k$ tribus forment la famille disjointe maximale). Les deux quantités coincident à chaque étage :

$$s(\text{tribes}_{k,2}) = bs(\text{tribes}_{k,2}) = k \quad \text{pour } k = 2, 3, 4 \text{ (mesuré)},$$

et la pente de $s$ seule continue linéairement ($s = 5, 6$ pour $k = 5, 6$). **Conclusion mesurée : les tribus seules ne séparent jamais globalement** — l'écart local $0 < k$ au sommet nul est systématiquement rattrapé ailleurs sur le cube. C'est l'echo, cette fois par une famille entière mesurée, de la réfutation du §5 : la séparation *globale* $bs > s$ exige d'autres constructions (l'adressage de Rubinstein 1995, dont la famille 132 du dépôt est le raffinement 2026). Le §6 teste exactement cette idée à l'échelle calculable.

## 6. Expérience 2 — la séparation globale naît à $n = 4$ (mesuré exhaustivement)

Deux questions, une méthode. **Question A** : la construction d'**adressage** — $k$ bits d'adresse, $2^k$ bits de données, $f(x) = x_{2^k + 2x_0 + x_1}$ pour $k = 2$ (le «Rubinstein en miniature») — sépare-t-elle déjà à $n = 6$ ? Intuition d'auteur : oui ($bs > s$ attendu). **Question B** : à quelle dimension $n$ **minimale** la séparation globale $bs(f) > s(f)$ existe-t-elle, ne serait-ce que sur une seule fonction ?

Les deux se tranchent par **balayage exhaustif** : à $n$ fixé, énumérer les $2^{2^n}$ fonctions booléennes totales et mesurer $s$ et $bs$ exacts de chacune (la DP du §5 rend le $bs$ exact abordable : $n = 3$ couvre 256 fonctions, $n = 4$ en couvre 65 536 — au-delà, l'énumération elle-même explose, la borne $2^{2^n}$ est intrinsèque).

In [7]:
# Experience 2 : (a) addressing addr6 mesure ; (b) balayage exhaustif n=3 puis n=4.
def addr6(x):  # 2 bits d'adresse (x0, x1) + 4 bits de donnees (x2..x5)
    return x[2 + 2 * x[0] + x[1]]

sa, ba = s(addr6, 6), bs(addr6, 6)
print(f"addr6 (n=6) : s global = {sa} | bs global = {ba} | ratio = {ba/sa:.2f}")

def s_table(ft, n, xi):
    return sum(1 for i in range(n) if ft[xi ^ (1 << i)] != ft[xi])

def bs_table(ft, n, xi):
    full = (1 << n) - 1
    sens = [B for B in range(1, full + 1) if ft[xi ^ B] != ft[xi]]
    by_low = {}
    for B in sens:
        by_low.setdefault(B & (-B), []).append(B)
    dp = [0] * (full + 1)
    for mask in range(1, full + 1):
        low = mask & (-mask)
        best = dp[mask ^ low]
        for B in by_low.get(low, ()):
            if B & mask == B:
                cand = 1 + dp[mask ^ B]
                if cand > best:
                    best = cand
        dp[mask] = best
    return dp[full]

for n in (3, 4):
    nv = 1 << n
    temoins = []
    for fid in range(1 << nv):
        ft = [(fid >> i) & 1 for i in range(nv)]
        sg = max(s_table(ft, n, xi) for xi in range(nv))
        bg = max(bs_table(ft, n, xi) for xi in range(nv))
        if bg > sg:
            temoins.append((fid, sg, bg))
    print(f"n={n} : fonctions avec bs>s global : {len(temoins)} sur {1 << nv}"
          + (f" -- paires (s,bs) distinctes : {sorted(set((sg, bg) for _, sg, bg in temoins))}"
             if temoins else ""))

# table de verite du premier temoin n=4
fid = 7128
ft = [(fid >> i) & 1 for i in range(16)]
print()
print("table du temoin fid=7128 (x3x2x1x0 -> f) :",
      {format(i, '04b'): ft[i] for i in range(16) if ft[i]})

addr6 (n=6) : s global = 3 | bs global = 3 | ratio = 1.00
n=3 : fonctions avec bs>s global : 0 sur 256


n=4 : fonctions avec bs>s global : 24 sur 65536 -- paires (s,bs) distinctes : [(2, 3)]

table du temoin fid=7128 (x3x2x1x0 -> f) : {'0011': 1, '0100': 1, '0110': 1, '0111': 1, '1000': 1, '1001': 1, '1011': 1, '1100': 1}


### Lecture du résultat

**Question A — réfutée par la mesure.** `addr6` donne $s = 3$, $bs = 3$ : ratio 1, pas de séparation. L'intuition d'auteur (« l'adressage sépare dès $k=2$ ») est fausse — exactement la même leçon que la réfutation `decide` du §5, cette fois attrapée par la mesure Python après avoir survécu au design. L'adressage de Rubinstein sépare bien, mais à une échelle où la structure d'adresse a assez de marge ($k$ grand) — pas en miniature.

**Question B — la séparation globale naît à $n = 4$.** Le balayage exhaustif donne un énoncé complet et falsifiable :

- à $n = 3$ : **aucune** des 256 fonctions ne sépare ($bs \le s$ partout) ;
- à $n = 4$ : **24 fonctions sur 65 536**, toutes exactement avec $s = 2$, $bs = 3$.

Le premier témoin (`fid = 7128` dans l'ordre de l'énumération) est affiché par sa table de vérité — un objet fini, que le lecteur peut re-mesurer. Notons ce que ce résultat **n'est pas** : une borne sur les constructions de la littérature (Rubinstein, famille 132). Il dit seulement qu'à l'échelle calculable exhaustive, l'écart global $bs > s$ apparaît dès $n = 4$ et y reste modeste (ratio $3/2$) ; la séparation superquadratique $bs \ge s^{2+\epsilon}$ de la famille 132 vit nécessairement à bien plus grande dimension — c'est cité avec son statut au §8, pas re-produit ici.

Enfin, la coexistence est instructive : `orOfAnds` — **nôtre**, $n = 4$, prouvée au lac — a $bs = s = 2$ globalement : elle n'est **pas** parmi les 24. Les 24 témoins sont ses voisines dans l'espace des fonctions, et la preuve formelle du lac (séparation locale) reste la seule chose qu'on peut certifier à coût raisonnable.

## 7. Friction et chemin de découverte

Ce que la construction de l'ajout 2026 a réellement coûté — consigné parce que c'est la partie réutilisable :

1. **La réfutation `decide` (l'incident fondateur du fichier).** Le premier design de `Sens2026.lean` affirmait $\forall x,\ s(\texttt{tribes32}, x) \le 2$ (« deux lampes par tribu, donc sensibilité au plus 2 »). Le noyau a rejeté l'énoncé **à la compilation** : au sommet $(0,1,0,1,0,1)$ — une lampe allumée par tribu — chaque **première** coordonnée allume sa conjonction à elle seule, donc $s = 3$. L'énoncé corrigé est devenu le lemme contre-intuitif `tribes32_sensitivity_eq_three`, et le fichier enseigne exactement ce renversement : *l'intuition ne survit pas à l'énumération exhaustive du noyau*.
2. **`set_option maxRecDepth` se place AVANT la docstring** — entre la docstring et le `theorem`, c'est une erreur de parse (`unexpected token 'set_option'`).
3. **`3 ≤ bs` ne s'unifie pas avec le but `0 < bs` via `exact`** — il faut `Nat.lt_of_lt_of_le (by decide) h` ; le `decide` seul dépasse la profondeur par défaut sur les `flipBlock` imbriqués (d'où le `maxRecDepth 8192`).
4. **L'adressage à 2 bits exige 4 bits de données** ($2^k$ bits pour $k$ bits d'adresse) : un premier `addr4` cassait avec `IndexError` — la dimension correcte est $n = k + 2^k = 6$.
5. **$bs$ exact : la récursion naïve set-packing explose à $n = 6$** (interruption au bout de plusieurs minutes) ; la DP sur sous-ensembles du §5 rend le même chiffre exact en une fraction de seconde. Un instrument se juge par ses faux négatifs : ici, la récursion « marchait » et mentait par le temps.
6. **Le PATH WSL** : `lake` n'est pas dans le PATH d'un bash non-login — `export PATH=$HOME/.elan/bin:$PATH` en tête de chaque commande (leçon déjà consignée au carnet 05, toujours vraie).
7. **L'hypothèse addr6 réfutée par la mesure** (§6, question A) : une intuition d'auteur qui avait survécu au design et mort à l'instrument — le même geste que la réfutation `decide`, appliqué à soi-même.

## 8. La veille 2026 — le dépôt openai/math, cité avec son statut

Le dépôt `openai/math` (publié 2026-10-06, Apache-2.0, 719 manuscrits / 372 familles dont 242 à squelette Lean publié) est une **source tierce non vérifiée par nous**. L'Epic Origami (#19898) impose une discipline d'écriture à trois statuts, inscrite dans chaque carnet :

| Statut de la famille | Ce que le carnet écrit |
|---|---|
| **formalisée** (squelette Lean publié) | « énoncé formalisé par les auteurs (`lean/docs/NNN.md`) ; confronté à notre formalisation » |
| **non formalisée** | « énoncé tel qu'annoncé, **non formalisé** en amont ; nous n'en faisons pas un résultat établi » |
| **nôtre** | « prouvé dans notre lac `X` (`#check`, `#print axioms`) » |

Jamais « OpenAI a prouvé X, donc X ». Les quatre familles que notre lac rencontre littéralement :

| Famille | Énoncé (titre du dépôt) | Statut |
|---|---|---|
| **127** | *Average sensitivity of polynomial threshold functions* — une PTF de degré $\le d$ sur le cube a une sensibilité moyenne $\le 8d\sqrt{n}$ (Gotsman–Linial) | **formalisée** (`lean/docs/127.md`) |
| **132** | *A superquadratic separation of sensitivity and block sensitivity* — il existe $f$ totale avec $bs(f) \ge s(f)^\alpha$ pour un $\alpha > 2$ fixé | **formalisée** (`lean/docs/132.md`) ; préprint 2026 **non relu par des experts** |
| **186** | *Uniform influence and sharp thresholds for graph and hypergraph properties* — Friedgut–Kalai : largeur de seuil $O((\log n)^{-2})$ (graphes) | **formalisée** (`lean/docs/186.md`) |
| **192** | *Boolean functions violate the square-root degree bound by arbitrary factors* — pour tout $C > 0$ il existe $f$ signe avec $\sum_i \hat{f}(\{i\}) > C\sqrt{\deg(f)}$ | **formalisée** (`lean/docs/192.md`) |

Ce que ces familles changent pour **notre** chaîne (§2) : la famille **132** clôt le débat sur l'exposant par en bas — le renforcement quadratique $bs \le C \cdot s^2$ est **faux** ; l'exposant optimal vit donc strictement entre 2 et 4. La famille **127** est l'analogue lisse de la sensibilité (PTF), la **186** le pont vers les seuils de percolation, la **192** la version Fourier (les coefficients de niveau 1 que notre lac définit dans `Fourier.lean`). Aucune de ces quatre n'est re-prouvée ici : leur échelle dépasse le lac, et ce carnet préfère mesurer exhaustivement ce qui est calculable (§5–6) que maquiller une citation en résultat.

## 9. Exercices

Trois exercices pour s'approprier les définitions et les instruments du §5. Chaque cellule s'exécute **telle quelle** — le travail est dans les `TODO`.

In [8]:
# Exercice 1 -- ET logique sur 3 bits : mesurer la sensibilite locale (a completer)
def and3(x): return x[0] and x[1] and x[2]

# Etape 1 : implementer sens_locale(f, x) -- le nombre de coordonnees dont le
# retournement (une a une) change f(x). Indice : s'inspirer de s_at du paragraphe 5.
def sens_locale(f, x):
    # TODO etudiant
    return None

# Etape 2 : mesurer s local de and3 au sommet 111, puis au sommet 011.
# Question attendue : pourquoi s(and3, 111) = 3 mais s(and3, 011) = 1 ?
print("Exercice a completer")

Exercice a completer


### Exercice 2 — Tribu asymétrique : prédire, puis mesurer

La famille `tribes_kt(k, t)` du §5 a construit des tribus **symétriques** ($t = 2$). La théorie (Nisan–Szegedy) prédit $s(\text{tribes}_{k,t}) = \max(k, t)$. Pour $k = 2$, $t = 3$ ($n = 6$) : prédire la valeur **avant** de mesurer, puis confronter — le geste exact du carnet.

In [9]:
# Exercice 2 -- tribes k=2, t=3 (n=6) : PREDIRE puis mesurer (a completer)
# Etape 1 : ecrire ici la prediction en commentaire, puis la coder.
# Indice : s(tribes_{k,t}) = max(k, t) ; que vaut max(2, 3) ?
prediction = None  # TODO etudiant : remplacer par la valeur attendue

# Etape 2 : construire la fonction avec tribes_kt(2, 3) (paragraphe 5) et
# mesurer s global avec l'instrument s(f, n). Attention : la cellule du
# paragraphe 5 doit avoir ete executee pour que tribes_kt existe.
# Etape 3 : comparer prediction et mesure -- si elles different, trouver le
# sommet qui realise le max (le meme geste que x_impair au paragraphe 5).
print("Exercice a completer")

Exercice a completer


### Exercice 3 — Où vit l'écart du témoin $n = 4$ ?

Le balayage du §6 a trouvé 24 fonctions de $Q_4$ avec $bs > s$ global, toutes $(s, bs) = (2, 3)$. La table du premier témoin est donnée. L'écart global est réalisé en des sommets précis — les trouver.

In [10]:
# Exercice 3 -- localiser les sommets separatifs du temoin fid=7128 (a completer)
fid, n = 7128, 4
table = [(fid >> i) & 1 for i in range(16)]
def f_temoin(x):  # la fonction du temoin, depuis sa table
    return table[sum(b << i for i, b in enumerate(x))]

# Etape 1 : pour chacun des 16 sommets, afficher s local et bs local
# (instruments s_at / bs_at du paragraphe 5).
# Etape 2 : isoler les sommets ou bs local = 3. Que vaut s local sur chacun ?
# Indice : l'ecart global (2, 3) ne signifie pas bs - s = 1 en TOUT sommet.
print("Exercice a completer")

Exercice a completer


## 10. Conclusion

### 10.1 Le corps du carnet

- **L'énoncé** — sensibilité, sensibilité de bloc, la conjecture 1992–2019 (§1) et sa chaîne en trois maillons dont un seul à nous (§2).
- **Le socle vérifié** — audit du pin `v4.33.0` : rien dans Mathlib propre, l'Archive à la racine du paquet ; le lac `sensitivity_lean` est le dust-off autonome de Huang, compilé ici, `0 sorry` au compteur canonique, onze signatures et sept `#print axioms` aux trois axiomes standard (§3) — le statut **« nôtre »** démontré.
- **L'ajout 2026** — `Sens2026.lean` : le témoin `tribes32`, quatre théorèmes dont le lemme contre-intuitif $s(x^{\text{impair}}) = 3$ que `decide` a imposé contre l'intuition d'auteur (§5).
- **Les mesures exhaustives** — la DP sous-ensembles pour $bs$ exact ; la mesure Python reproduit le `decide` chiffre à chiffre ; l'arc des tribus ratio 1 (elles ne séparent jamais globalement, $k = 2, 3, 4$ mesuré) ; l'hypothèse d'adressage réfutée à $n = 6$ ; la séparation globale $bs > s$ **naît à $n = 4$** — 0 témoin sur 256 fonctions à $n = 3$, 24 sur 65 536 à $n = 4$, toutes $(2, 3)$ (§5–6).
- **La veille avec statut** — familles 127, 132, 186, 192 du dépôt openai/math, toutes citées *formalisées en amont*, jamais adoptées comme résultats établis ; 132 borne l'exposant de la chaîne par en bas (le renforcement quadratique est faux), l'exposant optimal reste dans la bande ouverte $(2, 4]$ (§8).

### 10.2 Ce que ce carnet ne prétend pas

Il ne prouve ni l'équivalence de Gotsman–Linial, ni la borne Nisan–Szegedy, ni aucun énoncé des familles citées ; il ne reproduit aucune construction superquadratique. Tout ce qui est mesuré est exhaustif, exact et re-exécutable ; tout ce qui est cité porte son statut.

### 10.3 Pour aller plus loin

Le carnet [Lean-12](../Lean-12-Sensitivity-Theorem.ipynb) développe la preuve spectrale de Huang ; le [12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) la variation constructiviste ; l'exercice 3 du §9 ouvre la voie à la formalisation d'un des 24 témoins $n = 4$ — un candidat naturel pour un prochain ajout additif au lac.